# Tutorial 1: Adam + $\rho$ the Exact Way, Then the Earlier Approximation

This notebook explains two different ways to combine Adam with a convergence-radius controller.

The main lesson is that the **exact controller** should act on Adam's **actual proposed step**:
- let Adam propose an update direction and step norm,
- compute $\rho_a(v)$ along that proposed direction,
- rescale the step only if the proposal is too large relative to $\rho_a$.

After that, we compare it to an **earlier simpler approximation** used in exploratory work:
- compute $\rho$ from the current logits,
- use it only to scale Adam's global learning rate,
- then let Adam step as usual.

Both methods use the same model, data, and base learning rates. The key question is whether they control the **actual normalized step size**
$r = \tau / \rho_a$ in the same way.


In [ ]:

from pathlib import Path
import sys


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (base / "tutorials").exists():
            return base
    raise RuntimeError("Run this notebook from inside the ghosts-of-softmax repository.")


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repo root: {REPO}")



## 1. Setup

We keep this tutorial small and deliberate:
- one architecture (`MLP`),
- one seed,
- three Adam base learning rates,
- one optional multiseed block at the end.

The three base learning rates play different roles:
- `0.001`: safely below the radius limit,
- `0.5`: large enough that the exact controller should intervene,
- `5.0`: intentionally absurd, so the difference between the two controllers becomes obvious.


In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.func import functional_call, jvp
from torch.utils.data import DataLoader, TensorDataset

from ghosts.control import RhoScaledAdam

torch.set_num_threads(1)

DEVICE = torch.device("cpu")
LOW_LR = 0.001
MID_LR = 0.5
HIGH_LR = 5.0
LRS = [LOW_LR, MID_LR, HIGH_LR]
EPOCHS = 20
BATCH_SIZE = 128
TARGET_R = 1.0
RHO_CAP = 1.0
RHO_FLOOR = 1e-3
SEED = 7

RUN_MULTI_SEED = False
MULTI_SEEDS = [0, 1, 2, 3, 4]

PALETTE = {
    "exact": "#006BA2",
    "scaled": "#E3120B",
    "dark": "#3D3D3D",
}


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)



## 2. Model and data

As in Tutorial 0, we use the `sklearn` digits dataset and a small MLP. That keeps the comparison focused on the controller logic rather than on architectural details.


In [ ]:

class MLP(nn.Module):
    def __init__(self, width=128):
        super().__init__()
        self.fc1 = nn.Linear(64, width)
        self.fc2 = nn.Linear(width, width)
        self.fc3 = nn.Linear(width, 10)

    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = F.gelu(self.fc2(x))
        return self.fc3(x)


def make_model():
    return MLP().to(DEVICE)


def build_digits_loaders(seed: int, batch_size: int = 128):
    digits = load_digits()
    X = digits.data.astype(np.float32) / 16.0
    y = digits.target.astype(np.int64)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y
    )
    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
    return train_loader, test_loader


train_loader, test_loader = build_digits_loaders(SEED, BATCH_SIZE)
len(train_loader), len(test_loader)



    ## 3. The exact Adam + $
ho$ controller

    The correct geometric rule is:

    1. let Adam propose its next update direction,
    2. measure that proposal's step norm,
    3. compute $
ho_a(v)$ along **that same direction**,
    4. rescale only if the proposed step is too long.

    In symbols, if Adam proposes a unit-learning-rate update vector `u_unit`, then

    ```python
    tau_raw = base_lr * ||u_unit||
    v = -u_unit / ||u_unit||
    rho_batch = rho_a(v)
    lr_eff = min(base_lr, target_r * rho_batch / ||u_unit||)
    ```

    This is the optimizer-agnostic form of the controller:
    - the base optimizer proposes a step,
    - the controller checks that step against the local radius,
    - the direction stays the optimizer's direction,
    - only the length changes.



    ## 4. The earlier simpler approximation

    Earlier exploratory experiments often used an easier shortcut: scale Adam's global learning rate by a scalar `rho` computed from the current logits.

    ```python
    rho_batch = computeRho(logits)
    scale = clamp(rho_batch, rhoFloor, rhoCap)
    lr_eff = base_lr * scale
    ```

    This is cheaper to package as an optimizer, but it is **not** the exact controller above.

    Why not?
    - it does not compute $
ho_a$ along Adam's actual proposed direction,
    - it does not compare the actual proposed step norm against that directional radius,
    - so it does not directly enforce $r = \tau / \rho_a \le 1$.

    The purpose of this notebook is to show that distinction directly.


In [ ]:

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            total_loss += float(loss.item()) * len(xb)
            total_correct += int((logits.argmax(dim=1) == yb).sum())
            total_count += len(xb)
    return total_loss / total_count, total_correct / total_count


def adam_unit_step(model, opt):
    group_by_param = {}
    for group in opt.param_groups:
        for p in group['params']:
            group_by_param[p] = group

    step_vecs = []
    sq = 0.0
    for p in model.parameters():
        if p not in group_by_param or p.grad is None:
            z = torch.zeros(p.numel(), device=p.device, dtype=p.dtype)
            step_vecs.append(z)
            continue

        group = group_by_param[p]
        beta1, beta2 = group['betas']
        eps = float(group['eps'])
        wd = float(group.get('weight_decay', 0.0))
        maximize = bool(group.get('maximize', False))

        g = p.grad.detach()
        if maximize:
            g = -g

        state = opt.state.get(p, {})
        if 'exp_avg' in state:
            m_prev = state['exp_avg']
            v_prev = state['exp_avg_sq']
            step_prev_obj = state.get('step', 0)
            step_prev = int(step_prev_obj.item()) if torch.is_tensor(step_prev_obj) else int(step_prev_obj)
        else:
            m_prev = torch.zeros_like(p)
            v_prev = torch.zeros_like(p)
            step_prev = 0

        step_t = step_prev + 1
        m_t = beta1 * m_prev + (1.0 - beta1) * g
        v_t = beta2 * v_prev + (1.0 - beta2) * (g * g)
        bc1 = 1.0 - (beta1 ** step_t)
        bc2 = 1.0 - (beta2 ** step_t)
        denom = v_t.sqrt() / math.sqrt(max(bc2, 1e-16)) + eps
        adam_term = (m_t / max(bc1, 1e-16)) / denom
        update = adam_term
        if wd != 0.0:
            update = update + wd * p.detach()

        step_vecs.append(update.flatten())
        sq += float((update * update).sum().item())

    step_vec = torch.cat(step_vecs)
    norm = math.sqrt(sq)
    return step_vec, norm


def batch_rho_jvp(model, xb, v_flat):
    params = dict(model.named_parameters())
    tangents = {}
    offset = 0
    for name, p in params.items():
        numel = p.numel()
        tangents[name] = v_flat[offset:offset + numel].view_as(p)
        offset += numel

    def fwd(pdict):
        return functional_call(model, pdict, (xb,))

    was_training = model.training
    model.eval()
    _, dlogits = jvp(fwd, (params,), (tangents,))
    if was_training:
        model.train()

    spread = dlogits.max(dim=1).values - dlogits.min(dim=1).values
    delta_a = float(spread.max().item())
    return math.pi / max(delta_a, 1e-12)


def run_training(mode: str, base_lr: float, seed: int):
    set_seed(seed)
    train_loader, test_loader = build_digits_loaders(seed, BATCH_SIZE)
    model = make_model()
    if mode == 'scaled':
        opt = RhoScaledAdam(model.parameters(), lr=base_lr, rhoCap=RHO_CAP, rhoFloor=RHO_FLOOR)
    elif mode == 'exact':
        opt = torch.optim.Adam(model.parameters(), lr=base_lr)
    else:
        raise ValueError(mode)

    history = {
        'train_loss': [],
        'test_acc': [],
        'max_r': [],
        'mean_r': [],
        'mean_eff_lr': [],
        'mean_rho': [],
    }

    for _ in range(EPOCHS):
        model.train()
        batch_losses = []
        batch_r = []
        batch_eff_lr = []
        batch_rho = []

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()

            unit_step_vec, unit_norm = adam_unit_step(model, opt)
            if unit_norm < 1e-12:
                if mode == 'scaled':
                    opt.step(logits=logits.detach())
                    eff_lr = float(opt.getStats()['effectiveLR'])
                else:
                    opt.step()
                    eff_lr = float(base_lr)
                rho_a = float('inf')
                r = 0.0
            else:
                v_dir = -unit_step_vec / unit_norm
                rho_a = batch_rho_jvp(model, xb, v_dir)
                if mode == 'exact':
                    eff_lr = min(base_lr, TARGET_R * rho_a / unit_norm)
                    for group in opt.param_groups:
                        group['lr'] = eff_lr
                    opt.step()
                else:
                    opt.step(logits=logits.detach())
                    eff_lr = float(opt.getStats()['effectiveLR'])

                tau = eff_lr * unit_norm
                r = tau / rho_a if rho_a > 0 else float('inf')

            batch_losses.append(float(loss.item()))
            batch_r.append(float(r))
            batch_eff_lr.append(float(eff_lr))
            batch_rho.append(float(rho_a if math.isfinite(rho_a) else 0.0))

        _, test_acc = evaluate(model, test_loader)
        history['train_loss'].append(float(np.mean(batch_losses)))
        history['test_acc'].append(float(test_acc))
        history['max_r'].append(float(np.max(batch_r)))
        history['mean_r'].append(float(np.mean(batch_r)))
        history['mean_eff_lr'].append(float(np.mean(batch_eff_lr)))
        history['mean_rho'].append(float(np.mean(batch_rho)))

    return history


RUN_SPECS = [
    {'key': 'exact_low', 'mode': 'exact', 'base_lr': LOW_LR, 'label': f'exact Adam+rho (LR={LOW_LR})'},
    {'key': 'exact_mid', 'mode': 'exact', 'base_lr': MID_LR, 'label': f'exact Adam+rho (LR={MID_LR})'},
    {'key': 'exact_high', 'mode': 'exact', 'base_lr': HIGH_LR, 'label': f'exact Adam+rho (LR={HIGH_LR})'},
    {'key': 'scaled_low', 'mode': 'scaled', 'base_lr': LOW_LR, 'label': f'earlier rho-scaled Adam (LR={LOW_LR})'},
    {'key': 'scaled_mid', 'mode': 'scaled', 'base_lr': MID_LR, 'label': f'earlier rho-scaled Adam (LR={MID_LR})'},
    {'key': 'scaled_high', 'mode': 'scaled', 'base_lr': HIGH_LR, 'label': f'earlier rho-scaled Adam (LR={HIGH_LR})'},
]


def final_summary_row(spec, hist):
    return {
        'run': spec['label'],
        'final_acc': hist['test_acc'][-1],
        'peak_r': max(hist['max_r']),
        'final_mean_r': hist['mean_r'][-1],
        'final_mean_eff_lr': hist['mean_eff_lr'][-1],
        'final_mean_rho': hist['mean_rho'][-1],
    }



## 5. Single-seed run

We now run both controllers on the same model and split.

The comparison to watch is straightforward:
- the **exact controller** should drive the realized step ratio toward $r \approx 1$ once the base learning rate is large enough to matter,
- the **earlier rho-scaled alternative** may still take realized steps with $r \gg 1$ because it only scales Adam's global learning rate.


In [ ]:

single_seed = {}
for spec in RUN_SPECS:
    single_seed[spec['key']] = run_training(spec['mode'], spec['base_lr'], SEED)

summary_rows = [final_summary_row(spec, single_seed[spec['key']]) for spec in RUN_SPECS]
pd.DataFrame(summary_rows)



## 6. First look: loss and accuracy

Start with the most legible outputs:
- training loss on a log scale,
- test accuracy on a linear scale.

Reading key:
- **color = controller**,
- **line style = base learning rate**,
- **markers help when curves overlap**.


In [ ]:

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica Neue', 'DejaVu Sans'],
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

epochs = np.arange(1, EPOCHS + 1)
style_map = {
    'exact_low': {'color': PALETTE['exact'], 'ls': '-', 'marker': 'o', 'alpha': 0.82},
    'exact_mid': {'color': PALETTE['exact'], 'ls': '--', 'marker': 'o', 'alpha': 0.82},
    'exact_high': {'color': PALETTE['exact'], 'ls': '-.', 'marker': 'o', 'alpha': 0.82},
    'scaled_low': {'color': PALETTE['scaled'], 'ls': '-', 'marker': 's', 'alpha': 0.82},
    'scaled_mid': {'color': PALETTE['scaled'], 'ls': '--', 'marker': 's', 'alpha': 0.82},
    'scaled_high': {'color': PALETTE['scaled'], 'ls': '-.', 'marker': 's', 'alpha': 0.82},
}

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharex=True)

for spec in RUN_SPECS:
    key = spec['key']
    hist = single_seed[key]
    style = style_map[key]
    axes[0].semilogy(
        epochs, hist['train_loss'], label=spec['label'],
        color=style['color'], ls=style['ls'], lw=2.2,
        marker=style['marker'], ms=4.5, markevery=2, alpha=style['alpha']
    )
    axes[1].plot(
        epochs, hist['test_acc'], label=spec['label'],
        color=style['color'], ls=style['ls'], lw=2.2,
        marker=style['marker'], ms=4.5, markevery=2, alpha=style['alpha']
    )

axes[0].set_title('Training loss')
axes[1].set_title('Test accuracy')
axes[0].set_ylabel('loss')
axes[1].set_ylabel('accuracy')
axes[1].set_ylim(0.0, 1.02)

for ax in axes:
    ax.set_xlabel('epoch')
    ax.grid(True, alpha=0.25)

axes[0].legend(frameon=False, fontsize=8, ncol=2, loc='upper right')
fig.suptitle('Exact directional Adam controller versus earlier rho-scaled alternative', y=1.02, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()



## 7. Diagnostic view: effective learning rate and realized step ratio

Now inspect the controller quantities directly.

The left panel shows the mean effective learning rate used over each epoch.
The right panel shows the realized normalized step size
$r = \tau / \rho_a$ measured from the **actual Adam update direction**.

This is the most important panel in the notebook: the exact controller should lock onto the geometric target, while the earlier approximation may drift far above it.


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharex=True)

for spec in RUN_SPECS:
    key = spec['key']
    hist = single_seed[key]
    style = style_map[key]
    axes[0].semilogy(
        epochs, hist['mean_eff_lr'], label=spec['label'],
        color=style['color'], ls=style['ls'], lw=2.1,
        marker=style['marker'], ms=4.0, markevery=2, alpha=0.72
    )
    axes[1].semilogy(
        epochs, hist['max_r'], label=spec['label'],
        color=style['color'], ls=style['ls'], lw=2.1,
        marker=style['marker'], ms=4.0, markevery=2, alpha=0.72
    )

axes[0].set_title('Effective learning rate')
axes[1].set_title('Realized step ratio (max r per epoch)')
axes[0].set_ylabel('effective lr')
axes[1].set_ylabel('max r')
axes[1].axhline(TARGET_R, color=PALETTE['dark'], ls=':', lw=1.2)

for ax in axes:
    ax.set_xlabel('epoch')
    ax.grid(True, alpha=0.25, which='both')

axes[0].legend(frameon=False, fontsize=8, ncol=2, loc='upper right')
fig.suptitle('Controller diagnostics', y=1.02, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()



## 8. Summary table

The figure gives the visual story. The table below shows the same comparison numerically.


In [ ]:

summary_df = pd.DataFrame(summary_rows)
for col in ['final_acc', 'peak_r', 'final_mean_r', 'final_mean_eff_lr', 'final_mean_rho']:
    summary_df[col] = summary_df[col].map(lambda x: round(float(x), 6) if col in ['final_mean_eff_lr', 'final_mean_rho'] else round(float(x), 3))
summary_df



## 9. Optional: multiseed check

If you want a more stable comparison, flip `RUN_MULTI_SEED = True` and rerun the next cell.
The aggregated table reports mean final accuracy and median peak step ratio across seeds.


In [ ]:

if RUN_MULTI_SEED:
    multi_rows = []
    for seed in MULTI_SEEDS:
        for spec in RUN_SPECS:
            hist = run_training(spec['mode'], spec['base_lr'], seed)
            multi_rows.append({
                'run': spec['label'],
                'seed': seed,
                'final_acc': hist['test_acc'][-1],
                'peak_r': max(hist['max_r']),
            })

    multi_df = pd.DataFrame(multi_rows)
    display(
        multi_df.groupby('run', as_index=False)
        .agg(final_acc_mean=('final_acc', 'mean'), final_acc_std=('final_acc', 'std'), peak_r_median=('peak_r', 'median'))
        .round(3)
    )
else:
    print('Set RUN_MULTI_SEED = True to execute the multiseed check.')



    ## 10. What to remember

    - The **correct** way to apply Adam + $
ho$ is to compute $
ho_a$ along Adam's **actual proposed step direction**.
    - The controller should compare the **actual proposed step norm** to that radius and rescale only if needed.
    - A simpler rho-scaled learning-rate rule can still be useful as a heuristic, but it is not the same geometric controller and does not automatically keep $r = 	au / 
ho_a$ near $1$.

    If you want the same geometric controller for another optimizer, the pattern is the same:
    1. let the optimizer propose an update,
    2. extract its direction and norm,
    3. compute $
ho_a$ along that direction,
    4. shrink the step only if it exceeds the target radius.
